In [1]:
%cd ..

/home/blanka/Multi-Domain-Pruning


## Load & print sample

In [2]:
import pickle
import os
import pandas as pd
from IPython.display import display

sample_label_path = "/data/blanka/DATASETS/SPN/YOLOv8x_train70val30/validation/label"
sample_data_path = "/data/blanka/DATASETS/SPN/YOLOv8x_train70val30/validation/data"

sample = "162.pkl"

label_df = pd.read_pickle(os.path.join(sample_label_path, sample))
print(label_df.to_string())

state_df = pd.read_pickle(os.path.join(sample_data_path, sample))
print(state_df.to_string())

    recall  precision   map50   map90  n_params  recall_init  precision_init  map50_init  map90_init  n_params_init  n_layer_channels
73  0.0094     0.0004  0.0047  0.0015     49.99       0.8731           0.842      0.8749      0.6869          68.13             113.0
    alpha  is_pruned  in_ch  out_ch  kernel  stride  pad  n_pruned_ch
0     0.0          1      3      80       3       2    1            0
1     0.0          1     80     160       3       2    1            0
2     0.0          1    160      80       1       1    0            0
3     0.0          1    160      77       1       1    0            0
4     0.1          1    388     148       1       1    0           12
5     0.0          1     77      80       3       1    1            0
6     0.0          1     80      77       3       1    1            0
7     0.0          1     77      80       3       1    1            0
8     0.1          1     80      77       3       1    1            3
9     0.0          1     77     

## Encode & decode state/label

In [3]:
from state_predictor.coder import Coder
from utils.config_parser import ConfigParser

# Read and save config file
conf = ConfigParser.read("config/spn.ini")

coder = Coder(state_df, label_df, alpha_range=(0, 2.2))
filtered_state_df = state_df[conf.model.state_features]

encoded_state = coder.encode_state(filtered_state_df)
encoded_label_wo_norm = coder.encode_label(label_df, do_normalize=False)
encoded_label = coder.encode_label(label_df)

print(label_df)
print(encoded_label_wo_norm)
print(encoded_label)

    recall  precision   map50   map90  n_params  recall_init  precision_init  \
73  0.0094     0.0004  0.0047  0.0015     49.99       0.8731           0.842   

    map50_init  map90_init  n_params_init  n_layer_channels  
73      0.8749      0.6869          68.13             113.0  
tensor([0.2663, 0.9946])
tensor([-0.4675,  0.9893])


## Normalize & denormalize label

In [4]:
from state_predictor.spn import normalize, denormalize

dmap = -0.2
range = (0, 1)

norm_dmap = normalize(dmap, range)
print(norm_dmap)
denorm_dmap = denormalize(norm_dmap, range)
print(denorm_dmap)

-1.4
-0.19999999999999996


## Test SPN model (predict) for one sample

In [5]:
from src.model.spn_handler import SPNHandler

# load or define SPN model
spn_handler = SPNHandler(conf, run_name="20250412_183601_88a68f_optuna")
spn_handler.create(is_pretrained=True)
decoded_pred_spars, decoded_pred_dmap = spn_handler.predict(encoded_state.unsqueeze(dim=0)) # Returns DENORMALIZED VALUES !!!!

# Denormalize 
decoded_gt_spars, decoded_gt_dmap = denormalize(encoded_label, value_range=(0, 1))

# Calculate spacing based on the longest label
spacing = max(len(f"{decoded_gt_dmap:.4f}"), len(f"{decoded_pred_dmap:.4f}"), len(f"{decoded_gt_spars:.4f}"), len(f"{decoded_pred_spars:.4f}"))

# Print the values in the specified format with equal spacing
print(f"dmap:\n  gt:   {decoded_gt_dmap:>{spacing}.4f}\n  pred: {decoded_pred_dmap:>{spacing}.4f}")
print(f"\nspars:\n  gt:   {decoded_gt_spars:>{spacing}.4f}\n  pred: {decoded_pred_spars:>{spacing}.4f}")



dmap:
  gt:   0.9946
  pred: 0.4607

spars:
  gt:   0.2663
  pred: 0.2538


/home/blanka/Multi-Domain-Pruning/src/model/spn_handler.py:224: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu')


## Test SPN for all validation data

In [13]:
import torch
import matplotlib.pyplot as plt
from utils.config_parser import ConfigParser
from src.model.spn_handler import SPNHandler
from state_predictor.dataloader import create_pruning_dataloader


conf = ConfigParser.read("config/spn.ini")
val_dataloader = create_pruning_dataloader(conf, split_type="validation")

# load or define SPN model
spn_handler = SPNHandler(conf, run_name="20250412_183601_88a68f_optuna")
spn_handler.create(is_pretrained=True)

for batch_i, (data_gt, label_gt) in enumerate(val_dataloader): 

    for data_gti, label_gti in zip(data_gt, label_gt):

        denorm_pred_spars, denorm_pred_dmap = spn_handler.predict(data_gti.unsqueeze(dim=0))
        (denorm_gt_spars, denorm_gt_dmap) = denormalize(label_gti, value_range=(0, 1))

        # Calculate spacing based on the longest formatted number
        spacing = max(
            len(f"{decoded_gt_dmap:.4f}"),
            len(f"{decoded_pred_dmap:.4f}"),
            len(f"{decoded_gt_spars:.4f}"),
            len(f"{decoded_pred_spars:.4f}")
        )

        # Values
        print(f"{'spars':<6} {denorm_gt_spars:>{spacing}.4f} {denorm_pred_spars:>{spacing}.4f}")
        print(f"{'dmap':<6} {denorm_gt_dmap:>{spacing}.4f} {denorm_pred_dmap:>{spacing}.4f}\n")



/home/blanka/Multi-Domain-Pruning/src/model/spn_handler.py:224: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu')


spars  0.0614 0.0676
dmap   0.0249 -0.0111

spars  0.4267 0.4654
dmap   1.0000 0.7267

spars  0.1789 0.1778
dmap   0.1141 0.0130

spars  0.0418 0.0209
dmap   0.6342 -0.0585

spars  0.0066 -0.0012
dmap   0.0170 -0.0246

spars  0.2052 0.2032
dmap   0.4807 0.0623

spars  0.1110 0.0728
dmap   0.0322 -0.0142

spars  0.0618 0.0202
dmap   0.0419 -0.0548

spars  0.0809 0.0524
dmap   0.0078 -0.0268

spars  0.1243 0.1482
dmap   0.0733 0.0125

spars  0.0163 0.0101
dmap   0.1767 -0.0394

spars  0.0371 0.0497
dmap   0.9999 0.2483

spars  0.0038 0.0201
dmap   0.0043 -0.0486

spars  0.0750 0.0968
dmap   0.1231 0.0004

spars  0.1236 0.1586
dmap   0.1698 0.0164

spars  0.1361 0.1294
dmap   0.0666 0.0175

spars  0.2519 0.2096
dmap   0.6668 0.0365

spars  0.0000 0.0012
dmap   0.0000 -0.0277

spars  0.2209 0.1279
dmap   0.7917 0.0836

spars  0.0274 0.0416
dmap   0.2451 -0.0220

spars  0.2031 0.1645
dmap   0.2816 0.0286

spars  0.1411 0.0614
dmap   0.7891 0.0719

spars  0.0013 0.0121
dmap   0.0007 -0.0421
